In [0]:
# ── RUTAS
DELTA_NORMAL_PATH = "/Volumes/workspace/default/phisical_measures/delta_normal/"
DELTA_ATTACK_PATH = "/Volumes/workspace/default/phisical_measures/delta_attack/"

# ── LEER
df_normal = spark.read.format("delta").load(DELTA_NORMAL_PATH)
df_attack = spark.read.format("delta").load(DELTA_ATTACK_PATH)

# ── VERIFICAR
print("\n=== df_normal ===")
print(f"Filas: {df_normal.count():,}")
print(f"Columnas: {len(df_normal.columns)}")
df_normal.printSchema()
display(df_normal.limit(3))

print("\n=== df_attack ===")
print(f"Filas: {df_attack.count():,}")
print(f"Columnas: {len(df_attack.columns)}")
df_attack.printSchema()
display(df_attack.limit(3))

In [0]:
from pyspark.sql import functions as F

# ── RUTAS
DELTA_NORMAL_PATH = "/Volumes/workspace/default/phisical_measures/delta_normal/"
DELTA_ATTACK_PATH = "/Volumes/workspace/default/phisical_measures/delta_attack/"

# ── LEER
df_normal = spark.read.format("delta").load(DELTA_NORMAL_PATH)
df_attack = spark.read.format("delta").load(DELTA_ATTACK_PATH)

# ── CORREGIR TYPO — comparar directamente con "A ttack"
df_attack = df_attack.withColumn(
    "Normal_Attack",
    F.when(F.col("Normal_Attack") == "A ttack", "Attack")
     .otherwise(F.col("Normal_Attack"))
)

# ── VERIFICAR
display(
    df_attack.groupBy("Normal_Attack")
    .agg(F.count("*").alias("count"))
    .orderBy("count", ascending=False)
)

In [0]:
from pyspark.sql import functions as F
# ── NOMBRE DE LA COLUMNA DE ETIQUETA


LABEL_COL = "Normal_Attack"  # ajusta si el nombre es diferente tras la limpieza

# ── VALORES DISTINTOS EN CADA DF
print("=== df_normal ===")
display(
    df_normal.groupBy(LABEL_COL)
    .agg(F.count("*").alias("count"))
    .orderBy("count", ascending=False)
)

print("=== df_attack ===")
display(
    df_attack.groupBy(LABEL_COL)
    .agg(F.count("*").alias("count"))
    .orderBy("count", ascending=False)
)

In [0]:
from pyspark.sql import functions as F

# ── RANGO TEMPORAL DE CADA DF
print("=== RANGO TEMPORAL ===")
display(
    df_normal.agg(
        F.min("timestamp_dt").alias("normal_inicio"),
        F.max("timestamp_dt").alias("normal_fin")
    )
)

display(
    df_attack.agg(
        F.min("timestamp_dt").alias("attack_inicio"),
        F.max("timestamp_dt").alias("attack_fin")
    )
)

# ── TIMESTAMPS COINCIDENTES ENTRE LOS DOS DFs
ts_normal = df_normal.select("timestamp_dt").distinct()
ts_attack = df_attack.select("timestamp_dt").distinct()

coincidentes = ts_normal.join(ts_attack, on="timestamp_dt", how="inner")

print(f"\nTimestamps distintos en df_normal: {ts_normal.count():,}")
print(f"Timestamps distintos en df_attack: {ts_attack.count():,}")
print(f"Timestamps coincidentes:           {coincidentes.count():,}")

# ── VER LOS COINCIDENTES SI LOS HAY
if coincidentes.count() > 0:
    print("\n=== MUESTRA DE TIMESTAMPS COINCIDENTES ===")
    display(coincidentes.orderBy("timestamp_dt").limit(20))

In [0]:
# ── LEER DELTA DE FEATURES DE RED
DELTA_FEATURES_PATH = "/Volumes/workspace/default/network_data/features_delta/"

df_features = spark.read.format("delta").load(DELTA_FEATURES_PATH)

# ── VERIFICACIÓN BÁSICA
print(f"Total ventanas:  {df_features.count():,}")
print(f"Total columnas:  {len(df_features.columns)}")

print("\n=== RANGO TEMPORAL ===")
display(
    df_features.agg(
        F.min("window_start").alias("inicio"),
        F.max("window_end").alias("fin")
    )
)

print("\n=== SCHEMA ===")
df_features.printSchema()

display(df_features.limit(5))

In [0]:
# ── 1. UNIR df_normal Y df_attack EN UN SOLO DF CON LABEL
df_physical_labels = df_normal.select(
    F.col("timestamp_dt"),
    F.when(F.col("Normal_Attack") == "Attack", 1).otherwise(0).alias("label_physical")
).union(
    df_attack.select(
        F.col("timestamp_dt"),
        F.when(F.col("Normal_Attack") == "Attack", 1).otherwise(0).alias("label_physical")
    )
)

In [0]:
display(df_physical_labels.limit(5))
# diplay el numero de registros de df_physical_labels
print(df_physical_labels.count())
# imprimir el numero de ataques y el numero de registros sin ataques
print(df_physical_labels.filter(df_physical_labels.label_physical == 1).count())
print(df_physical_labels.filter(df_physical_labels.label_physical == 0).count())

In [0]:
# ── 3. JOIN CON df_features POR TIMESTAMP
df_features_updated = df_features \
    .drop("label") \
    .join(
        df_physical_labels,
        df_features["window_start"] == df_physical_labels["timestamp_dt"],
        how="left"
    ) \
    .drop("timestamp_dt") \
    .withColumnRenamed("label_physical", "label")

# ── 4. VERIFICAR RESULTADO
print("\n=== DISTRIBUCIÓN LABEL ACTUALIZADO ===")
display(
    df_features_updated.groupBy("label")
    .agg(F.count("*").alias("count"))
    .orderBy("label")
)

print(f"\nVentanas totales:      {df_features_updated.count():,}")
print(f"Ventanas con label:    {df_features_updated.filter(F.col('label').isNotNull()).count():,}")
print(f"Ventanas sin label:    {df_features_updated.filter(F.col('label').isNull()).count():,}")

In [0]:
# Ver qué aspecto tienen los timestamps de ambos lados
df_features.select("window_start").show(5, truncate=False)
df_physical_labels.select("timestamp_dt").show(5, truncate=False)

# Ver si los NULLs caen fuera del rango temporal de los físicos
display(
    df_features_updated.filter(F.col("label").isNull())
    .agg(
        F.min("window_start").alias("null_inicio"),
        F.max("window_start").alias("null_fin"),
        F.count("*").alias("count_null")
    )
)

# Comparar con el rango de los físicos
display(
    df_physical_labels.agg(
        F.min("timestamp_dt").alias("fisicos_inicio"),
        F.max("timestamp_dt").alias("fisicos_fin")
    )
)

In [0]:
from pyspark.sql import functions as F

FISICOS_INICIO = "2015-12-22 16:00:00"
FISICOS_FIN    = "2015-12-31 23:59:59"

df_features_final = df_features_updated \
    .filter(
        # Descartar ventanas fuera del rango físico
        (F.col("window_start") >= F.lit(FISICOS_INICIO)) &
        (F.col("window_start") <= F.lit(FISICOS_FIN))
    ) \
    .withColumn(
        # Los 81 NULLs dentro del rango → imputar como 0 (Normal)
        "label",
        F.coalesce(F.col("label"), F.lit(0))
    )

# Verificación
display(
    df_features_final.groupBy("label").count()
)

print(f"Total ventanas finales: {df_features_final.count():,}")

In [0]:
# ── 2. GUARDAR EN DELTA
df_features_final \
    .repartition(64) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_FEATURES_PATH)

print(f"✅ Dataset final guardado en: {DELTA_FEATURES_PATH}")

# ── 3. VERIFICACIÓN
df_check = spark.read.format("delta").load(DELTA_FEATURES_PATH)

total = df_check.count()
print(f"\nTotal ventanas: {total:,}")

display(
    df_check.groupBy("label")
    .count()
    .withColumn("porcentaje", F.round(F.col("count") / total * 100, 2))
    .orderBy("label")
)

# Comprobar que no quedan NULLs en label
nulls = df_check.filter(F.col("label").isNull()).count()
print(f"NULLs en label: {nulls}")